# 15 - Evaluate a Trained Model

Run this **after** a training notebook. This is the evaluation notebook: load a saved Mouse Core checkpoint and score it on live environments.

The inference flow is:

1. Build evaluation environments with `EnvConfig` and `make_group_env`.
2. Load a checkpoint with `load_model` on `MODEL_ID` and the packing spec with `load_tokenizer` on `TOKENIZER_ID` (a different Hub repo).
3. Step environments, convert the decode `ModelOutput` to actions with `model.get_action` (Q at the last valid head-output token), and carry the cache returned by `model(..., use_cache=True)`.
4. Record scores and optional render frames.

The rendering code is specific to this example environment; the model loading and cached inference pattern is the reusable Mouse Core part.


In [ ]:
import os

import torch
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

import gymnasium as gym
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv

ensure_registered()
from mouse_gym import EnvConfig, make_group_env
from mouse_core import load_model, load_tokenizer
from mouse_core.models.kv_policy import (
    cache_needs_rebuild,
    rebuild_starts,
    resolve_cache_bounds,
)

# FrozenLake renders via pygame; run headless in environments without a display.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

MODEL_ID = "mouse-example-model-offline"          # Hugging Face model repo for load_model
TOKENIZER_ID = "mouse-example-tokenizer-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
EVAL_STEPS = 512                                  # lockstep env steps for inference (passed to run_eval)
TASKS_PER_ENV = 1                                 # used only for video score formatting / frame cutoff
MAX_TASK_EPISODES = 20                            # max episodes per task
MAX_STEPS_PER_EPISODE = 30                        # max steps per episode
NUM_ENVS = 10                                     # number of environment streams in the GroupEnv
EVAL_SEED_OFFSET = 1_000_000                      # held-out env seed stream (far from train)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Build Evaluation Environment

The model was trained on rows containing `action`, `observation`, `reward`, `episode_terminated`, `episode_truncated`, and `episode_index` (emitted when `episode_start` is true). `terminated` and `truncated` are objective columns. The evaluation environment must emit the same fields. Rendering is optional and only used here to create replay videos.


In [ ]:
def make_lake(*, stream: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            map_seed=stream,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
            render_mode="rgb_array",
            emit_q_star=False,
        )
        return RepeatTaskEnv(
            lake,
            max_task_episodes=MAX_TASK_EPISODES,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )
    return env_fn

configs = [
    EnvConfig(
        name=f"proc_frozenlake_{i}",
        seed=i + EVAL_SEED_OFFSET,
        seed_mode="stream",
        env_fn=make_lake(stream=i + EVAL_SEED_OFFSET),
        reset_options={"regenerate_map": True},
    )
    for i in range(NUM_ENVS)
]

eval_env = make_group_env(configs)

def new_cursor(*, n: int) -> dict:
    """Per-stream task and episode counters for a mouse-gym group."""
    return {
        "task_index": [0] * n,
        "episode_index": [0] * n,
        "episode_start": [True] * n,
    }

def step_row(*, env, index: int, action, output, cursor: dict) -> dict:
    """Flatten one mouse-gym step of a RepeatTaskEnv into a datastore row.

    ``terminated`` / ``truncated`` are the mouse-gym flags (the task, when
    the wrapped env is a RepeatTaskEnv). Episode ends stay on the
    observation and are copied onto the row for the text stream.
    """
    observation, reward, terminated, truncated, info = output
    episode_terminated = bool(observation["episode_terminated"])
    episode_truncated = bool(observation["episode_truncated"])
    row = {
        "action": int(action),
        "observation": observation["observation"],
        "reward": float(reward),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "episode_terminated": episode_terminated,
        "episode_truncated": episode_truncated,
        "episode_ended": episode_terminated or episode_truncated,
        "episode_index": cursor["episode_index"][index],
        "episode_start": cursor["episode_start"][index],
        "task_index": cursor["task_index"][index],
        "step_index": env.envs[index].step_index,
    }
    if "q_star" in info:
        row["info_q_star"] = info["q_star"]
    if terminated or truncated:
        cursor["task_index"][index] += 1
        cursor["episode_index"][index] = 0
        cursor["episode_start"][index] = True
    elif episode_terminated or episode_truncated:
        cursor["episode_index"][index] += 1
        cursor["episode_start"][index] = True
    else:
        cursor["episode_start"][index] = False
    return row

cursor = new_cursor(n=len(eval_env.names))


## Load Model

`load_model` downloads a saved Mouse Core checkpoint and reconstructs the full `Model`, including the backbone (token embeddings included), heads, and weights. You do not need to rebuild the architecture by hand. Three things are settings for this machine rather than part of the checkpoint, so `load_model` asks for them: `train_kernel` (kernel of the uncached forward, `"flex"` = block-sparse FlexAttention), `decode_kernel` (kernel of cached decode, `"flex"` = paged FlexAttention) and `dtype` for the base weights (`torch.float32`, matching the training notebooks). `model.to(device)` then only moves.

The packing spec is not on the backbone. Training pushed `tokenizer.json` to a different Hub repo (`TOKENIZER_ID`); `load_tokenizer` on that repo is the one way to get formats, when-gates, `group_start`, and `head_output` back. A live HF tokenizer or image tokenizer is an execution choice — pass `tokenizer=` / `image_tokenizer=` when the spec needs them.


In [ ]:
model = load_model(
    repo_id_or_path=MODEL_ID,
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    force_download=True,
    map_location="cpu",
).eval()
model = model.to(device)

tokenizer = load_tokenizer(repo_id_or_path=TOKENIZER_ID, force_download=True)


## Batched Incremental Inference With FlexAttention (`run_eval`)

For live control, you usually want to process only the newest rows rather than the full history every step. Call the model with `use_cache=True` and pass the returned cache into the next call:

```python
kv_cache = None
out = model(tokenizer.pack_rows(rows=batch, continuing=None), cache=kv_cache, use_cache=True)
kv_cache = out.cache
actions = model.get_action(out=out, temperature=0.0, num_actions=num_actions)
```

Decode takes model inputs (`TokenBatch`): build them with `Tokenizer.pack_rows` — the same tokenizer used at train time. In lockstep environments, each stream can contribute one new step. Ragged per-stream lengths are preserved. `continuing` is required. Pass `None` on a full rebuild so group-start tokens emit once per sequence. Pass `True` per stream that already has cached tokens so the prefix is not emitted again.

A new `task_index` clears that stream's context and calls `cache.reset_rows`. Attention does not cross group ids. Completed-task scores are read from `env.metrics`.


In [ ]:
eval_contexts = [[] for _ in eval_env.names]

def run_eval(
    *,
    model,
    env,
    contexts: list,
    num_steps: int,
    max_cache: int = 512,
    start_cache=None,
    temperature: float = 0.0,
):
    """Greedy inference with frame capture; scores from ``env.metrics``.

    ``num_steps`` is lockstep ``GroupEnv`` steps for this call.
    Row ``contexts`` persist across the call (mutated in place). The KV cache
    is rebuilt from ``contexts`` at the start (then only on grow/``max_cache``).
    A new ``task_index`` clears that stream's context and resets its KV row.
    Completed-task scores come from ``env.metrics``.
    """
    model.eval()
    max_cache, start_cache = resolve_cache_bounds(max_cache=max_cache, start_cache=start_cache)
    n = len(env.names)
    if len(contexts) != n:
        raise ValueError(f"contexts has {len(contexts)} streams but env has {n}")
    kv_cache = None
    cached_starts = np.zeros(n, dtype=np.int64)
    cached_ends = np.zeros(n, dtype=np.int64)
    context_start = np.zeros(n, dtype=np.int64)
    for i, c in enumerate(contexts):
        if len(c) > max_cache:
            contexts[i] = c[-max_cache:]
    inputs = None
    outputs = None
    env.metrics.clear()
    num_actions = env.action_space.spaces[0].n
    video_names = env.names
    video_envs = env.envs
    frames_per_env = [[] for _ in video_names]
    steps_done = 0

    def _act_from_contexts(*, rebuild: bool) -> list[dict]:
        nonlocal kv_cache, cached_starts, cached_ends
        ends = np.array([len(c) for c in contexts], dtype=np.int64)
        need_rebuild = rebuild or cache_needs_rebuild(
            has_cache=kv_cache is not None,
            cached_starts=cached_starts,
            cached_ends=cached_ends,
            ends=ends,
            context_start=context_start,
            max_cache=max_cache,
            batch_complete=True,
        )
        with torch.no_grad():
            if need_rebuild:
                starts = rebuild_starts(
                    ends=ends,
                    context_start=context_start,
                    start_cache=start_cache,
                    max_cache=max_cache,
                )
                batch = [contexts[i][int(starts[i]) : int(ends[i])] for i in range(n)]
                out = model(
                    tokenizer.pack_rows(rows=batch, continuing=None),
                    use_cache=True,
                )
                kv_cache = out.cache
                cached_starts = starts
                cached_ends = ends.copy()
            else:
                batch = [
                    contexts[i][int(cached_ends[i]) : int(ends[i])] for i in range(n)
                ]
                # True when this stream already has cached tokens.
                continuing = [
                    bool(int(cached_ends[i]) > int(cached_starts[i]))
                    for i in range(n)
                ]
                out = model(
                    tokenizer.pack_rows(rows=batch, continuing=continuing),
                    cache=kv_cache,
                    use_cache=True,
                )
                kv_cache = out.cache
                cached_ends = ends.copy()
            actions = model.get_action(
                out=out, temperature=temperature, num_actions=num_actions
            )
        random_actions = env.action_space.sample()
        chosen = actions.cpu().numpy()
        return [
            chosen[i] if contexts[i] else random_actions[i]
            for i in range(n)
        ]

    for _ in range(num_steps):
        if outputs is None and any(contexts):
            inputs = _act_from_contexts(rebuild=True)
        elif any(contexts):
            inputs = _act_from_contexts(rebuild=False)
        else:
            inputs = env.action_space.sample()

        outputs = env.step(tuple(inputs))
        for i, (action, out) in enumerate(zip(inputs, outputs)):
            row = step_row(env=env, index=i, action=action, output=out, cursor=cursor)
            if contexts[i] and row["task_index"] != contexts[i][-1]["task_index"]:
                contexts[i] = []
                cached_starts[i] = 0
                cached_ends[i] = 0
                context_start[i] = 0
                if kv_cache is not None:
                    kv_cache.reset_rows(rows=[i])
            contexts[i].append(row)
        for i, video_env in enumerate(video_envs):
            if contexts[i][-1]["task_index"] >= TASKS_PER_ENV:
                continue
            frame = video_env.render()
            if frame is not None:
                frames_per_env[i].append(np.asarray(frame).copy())
        steps_done += 1
        if steps_done % 100 == 0:
            task_scores = [r for env_tasks in env.metrics.episode_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
            print(
                f"step {steps_done} | {len(task_scores)} tasks this window | "
                f"mean task score {mean_task_score:.2f}/{MAX_TASK_EPISODES}"
            )

    if kv_cache is not None:
        kv_cache.close()
    scores_per_env = [list(scores) for scores in env.metrics.episode_cum_rewards]
    return video_names, frames_per_env, scores_per_env

video_names, frames_per_env, eval_task_scores = run_eval(
    model=model, env=eval_env, contexts=eval_contexts, num_steps=EVAL_STEPS
)
eval_env.close()


## Replay MP4s

The remaining cells format environment-specific render frames as MP4s. This is not required for Mouse Core inference, but it is a useful way to inspect what the loaded policy is doing.


In [ ]:
def format_task_scores(*, task_scores):
    task_scores = task_scores[:TASKS_PER_ENV]
    scores_text = " ".join((f"{score:.0f}" for score in task_scores))
    return f"task scores [{scores_text}] / {MAX_TASK_EPISODES} | mean {sum(task_scores) / TASKS_PER_ENV:.2f}"

def display_env_replay(*, name, frames, task_scores, fps=5):
    print(f"{name}: {format_task_scores(task_scores=task_scores)}")
    if not frames:
        print("  (no frames captured)")
        return
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(w / 100, h / 100))
    ax.axis("off")
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)
    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))

for name, frames, task_scores in zip(video_names, frames_per_env, eval_task_scores):
    display_env_replay(name=name, frames=frames, task_scores=task_scores)
